- Dare informazione della posizione del pezzo -> [identificatore pezzo, identificatore slot del pezzo]: se tutte le coppie hanno elementi uguali e l'orientazione associata è ok allora il cubo è risolto
- Dare informazione dell'orientazione del pezzo -> 0 se il pezzo è sbagliato, 0.5 se il pezzo è orientato male ma nello slot giusto e 1 se il pezzo si trova nello slot con orientazione giusta

Praticamente esce fuori che sono 2 liste di dimensione uguale
- fare quella rappresentazione non mi genera direttamente le label se volessi fare una classificazione? oppure più precisamente se volessi fare un encoder?

- Rappresentare il cubo sotto forma di grafo con griglia 3x3x3:
  - `data.x`: feature del singolo pezzo -> vettore da 6 elementi cosi da coprire ogni faccia/coordinata (sembra che si perde informazione sull'orientazione)
  - `data.edge_index`: archi di ogni singolo pezzo
  - `data.pos`: posizione 3D del pezzo
  - 

In [1]:
import torch
import numpy as np
from magiccube import Cube
from magiccube.cube_base import Color, Face
import json

In [2]:
from torch_geometric.nn import GCNConv, GATConv
import networkx

/home/rainer/Code/DeepCube-RL/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
cube = Cube()
for coord, piece in cube.get_all_pieces().items():
    print(piece.get_piece_colors())

(<Color.O: 1>, <Color.Y: 3>, <Color.B: 4>)
(<Color.O: 1>, <Color.Y: 3>, None)
(<Color.O: 1>, <Color.Y: 3>, <Color.G: 5>)
(<Color.O: 1>, None, <Color.B: 4>)
(<Color.O: 1>, None, None)
(<Color.O: 1>, None, <Color.G: 5>)
(<Color.O: 1>, <Color.W: 2>, <Color.B: 4>)
(<Color.O: 1>, <Color.W: 2>, None)
(<Color.O: 1>, <Color.W: 2>, <Color.G: 5>)
(None, <Color.Y: 3>, <Color.B: 4>)
(None, <Color.Y: 3>, None)
(None, <Color.Y: 3>, <Color.G: 5>)
(None, None, <Color.B: 4>)
(None, None, <Color.G: 5>)
(None, <Color.W: 2>, <Color.B: 4>)
(None, <Color.W: 2>, None)
(None, <Color.W: 2>, <Color.G: 5>)
(<Color.R: 0>, <Color.Y: 3>, <Color.B: 4>)
(<Color.R: 0>, <Color.Y: 3>, None)
(<Color.R: 0>, <Color.Y: 3>, <Color.G: 5>)
(<Color.R: 0>, None, <Color.B: 4>)
(<Color.R: 0>, None, None)
(<Color.R: 0>, None, <Color.G: 5>)
(<Color.R: 0>, <Color.W: 2>, <Color.B: 4>)
(<Color.R: 0>, <Color.W: 2>, None)
(<Color.R: 0>, <Color.W: 2>, <Color.G: 5>)


In [4]:
cube.rotate("R U")
for id, elements in cube.get_all_faces().items():
    print(elements)

print(cube.get_all_pieces())

[[<Color.G: 5>, <Color.G: 5>, <Color.Y: 3>], [<Color.O: 1>, <Color.O: 1>, <Color.O: 1>], [<Color.O: 1>, <Color.O: 1>, <Color.O: 1>]]
[[<Color.W: 2>, <Color.B: 4>, <Color.B: 4>], [<Color.R: 0>, <Color.R: 0>, <Color.R: 0>], [<Color.R: 0>, <Color.R: 0>, <Color.R: 0>]]
[[<Color.Y: 3>, <Color.Y: 3>, <Color.B: 4>], [<Color.Y: 3>, <Color.Y: 3>, <Color.B: 4>], [<Color.Y: 3>, <Color.Y: 3>, <Color.B: 4>]]
[[<Color.W: 2>, <Color.W: 2>, <Color.W: 2>], [<Color.W: 2>, <Color.W: 2>, <Color.W: 2>], [<Color.G: 5>, <Color.G: 5>, <Color.G: 5>]]
[[<Color.O: 1>, <Color.O: 1>, <Color.O: 1>], [<Color.W: 2>, <Color.B: 4>, <Color.B: 4>], [<Color.W: 2>, <Color.B: 4>, <Color.B: 4>]]
[[<Color.R: 0>, <Color.R: 0>, <Color.R: 0>], [<Color.G: 5>, <Color.G: 5>, <Color.Y: 3>], [<Color.G: 5>, <Color.G: 5>, <Color.Y: 3>]]
{(0, 0, 0): OYB, (0, 0, 1): OY, (0, 0, 2): OYG, (0, 1, 0): OB, (0, 1, 1): O, (0, 1, 2): OG, (0, 2, 0): GWO, (0, 2, 1): GW, (0, 2, 2): YGR, (1, 0, 0): YB, (1, 0, 1): Y, (1, 0, 2): YG, (1, 1, 0): B, (1, 1

In [5]:
def get_other_orientations(piece_orientation: list):
    piece_type = len(piece_orientation)
    other_orientations = [piece_orientation.copy()]
    if piece_type == 3:
        for _ in range(2):
            piece_orientation.append(piece_orientation.pop(0))
            other_orientations.append(piece_orientation.copy())
    if piece_type == 2:
        other_orientations.append([piece_orientation[1], piece_orientation[0]])

    other_orientations.reverse()
    return other_orientations

In [6]:
print(get_other_orientations([1, 2, 3]))

[[3, 1, 2], [2, 3, 1], [1, 2, 3]]


In [12]:
for i, (coord, element) in enumerate(cube.get_all_pieces().items()):
    print(i, coord, element)

0 (0, 0, 0) OYB
1 (0, 0, 1) OY
2 (0, 0, 2) OYG
3 (0, 1, 0) OB
4 (0, 1, 1) O
5 (0, 1, 2) OG
6 (0, 2, 0) OWB
7 (0, 2, 1) OW
8 (0, 2, 2) OWG
9 (1, 0, 0) YB
10 (1, 0, 1) Y
11 (1, 0, 2) YG
12 (1, 1, 0) B
13 (1, 1, 2) G
14 (1, 2, 0) WB
15 (1, 2, 1) W
16 (1, 2, 2) WG
17 (2, 0, 0) RYB
18 (2, 0, 1) RY
19 (2, 0, 2) RYG
20 (2, 1, 0) RB
21 (2, 1, 1) R
22 (2, 1, 2) RG
23 (2, 2, 0) RWB
24 (2, 2, 1) RW
25 (2, 2, 2) RWG


In [8]:
# code for generating position label
# -1 -> no color
# R, O, W, Y, B, G -> 0, 1, 2, 3, 4, 5

# colors_map makes opposite color be recognized numerically 
colors_map = {
    2: -1,
    3: 1,
    4: -2,
    5: 2,
    0: -3,
    1: 3
}

cube = Cube()
position_label = {}
orientations_label = {}

for index, (coord, piece) in enumerate(cube.get_all_pieces().items()):
    #print(coord, piece, len(str(piece)))
    colors = []
    for color in piece.get_piece_colors():
        colors.append(0 if color is None else color.value)

    orientations_label[index] = get_other_orientations(colors)
    position_label[index] = colors

print(position_label)
print(orientations_label)

{0: [4, 1, 3], 1: [0, 1, 3], 2: [5, 1, 3], 3: [4, 1, 0], 4: [0, 1, 0], 5: [5, 1, 0], 6: [4, 1, 2], 7: [0, 1, 2], 8: [5, 1, 2], 9: [4, 0, 3], 10: [0, 0, 3], 11: [5, 0, 3], 12: [4, 0, 0], 13: [5, 0, 0], 14: [4, 0, 2], 15: [0, 0, 2], 16: [5, 0, 2], 17: [4, 0, 3], 18: [0, 0, 3], 19: [5, 0, 3], 20: [4, 0, 0], 21: [0, 0, 0], 22: [5, 0, 0], 23: [4, 0, 2], 24: [0, 0, 2], 25: [5, 0, 2]}
{0: [[4, 1, 3], [3, 4, 1], [1, 3, 4]], 1: [[0, 1, 3], [3, 0, 1], [1, 3, 0]], 2: [[5, 1, 3], [3, 5, 1], [1, 3, 5]], 3: [[4, 1, 0], [0, 4, 1], [1, 0, 4]], 4: [[0, 1, 0], [0, 0, 1], [1, 0, 0]], 5: [[5, 1, 0], [0, 5, 1], [1, 0, 5]], 6: [[4, 1, 2], [2, 4, 1], [1, 2, 4]], 7: [[0, 1, 2], [2, 0, 1], [1, 2, 0]], 8: [[5, 1, 2], [2, 5, 1], [1, 2, 5]], 9: [[4, 0, 3], [3, 4, 0], [0, 3, 4]], 10: [[0, 0, 3], [3, 0, 0], [0, 3, 0]], 11: [[5, 0, 3], [3, 5, 0], [0, 3, 5]], 12: [[4, 0, 0], [0, 4, 0], [0, 0, 4]], 13: [[5, 0, 0], [0, 5, 0], [0, 0, 5]], 14: [[4, 0, 2], [2, 4, 0], [0, 2, 4]], 15: [[0, 0, 2], [2, 0, 0], [0, 2, 0]], 16: 

In [9]:
for element in dir(cube):
    print(element)

__class__
__delattr__
__dir__
__doc__
__eq__
__format__
__ge__
__getattribute__
__gt__
__hash__
__init__
__init_subclass__
__le__
__lt__
__module__
__ne__
__new__
__reduce__
__reduce_ex__
__repr__
__setattr__
__sizeof__
__slots__
__str__
__subclasshook__
_cube
_cube_face_indexes
_cube_piece_indexes
_cube_piece_indexes_inv
_get_direction
_history
_is_outer_position
_move_to_slice
_rotate_once
_store_history
check_consistency
find_piece
generate_random_moves
get
get_all_faces
get_all_pieces
get_face
get_face_flat
get_kociemba_facelet_colors
get_kociemba_facelet_positions
get_piece
history
is_done
reset
reverse_history
rotate
scramble
set
size
undo
